# Model 1A, Experiment 02: Compact Engineered Decision Tree

This notebook evaluates a compact tree-oriented engineered representation for flights departing JFK. The target, `DepDel15`, equals 1 when departure delay is at least 15 minutes and 0 otherwise.

Experiment 02 keeps the same population, expanding-window 2019 folds, 96-configuration tree grid, metrics, training-only threshold procedure, and complete 2023 external development/validation dataset as Experiment 01. Only the feature manifest changes.

The tree-oriented manifest retains ordered raw schedule values and all six directional ASPM hourly counts, then adds selected features that would otherwise consume multiple tree levels to reconstruct: time-of-day and weekend groups, airline-destination interaction, scheduled-speed proxy, three-hour directional demand, congestion slopes and peak, weather transforms, wind components, condition summaries, and NOAA observation age. It excludes linear-model-specific cyclical pairs and high-cardinality flight identity.

All preprocessing remains inside each training fold. The 2023 labels are used only after the model and threshold choices are fixed, and the 2024 dataset is not loaded.


In [ ]:
# Parameters
AIRPORT = "JFK"
TRAIN_YEAR = 2019
VALIDATION_YEAR = 2023
TARGET = "DepDel15"

RANDOM_STATE = 42
N_TIME_SPLITS = 5
N_JOBS = 4


In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Load and validate the shared 2019 and 2023 departure feature data

The path lookup supports running from either the project root or the `models` directory. The explicit Experiment 02 allowlist selects a 34-feature tree-oriented subset from the shared raw and Appendix C engineered departure data.


In [ ]:
def find_project_root(start: Path, airport: str, years) -> Path:
    """Return the nearest parent containing every requested feature file."""
    relative_paths = [
        Path("data/features") / f"{airport}_{year}_departures.csv"
        for year in years
    ]
    for candidate in (start, *start.parents):
        if all((candidate / path).exists() for path in relative_paths):
            return candidate
    raise FileNotFoundError(f"Could not locate all required files: {relative_paths}")


PROJECT_ROOT = find_project_root(
    Path.cwd().resolve(),
    AIRPORT,
    [TRAIN_YEAR, VALIDATION_YEAR],
)
TRAIN_DATA_PATH = PROJECT_ROOT / "data/features" / f"{AIRPORT}_{TRAIN_YEAR}_departures.csv"
VALIDATION_DATA_PATH = (
    PROJECT_ROOT / "data/features" / f"{AIRPORT}_{VALIDATION_YEAR}_departures.csv"
)

train_df = pd.read_csv(TRAIN_DATA_PATH, low_memory=False)
validation_df = pd.read_csv(VALIDATION_DATA_PATH, low_memory=False)

print(f"Training source: {TRAIN_DATA_PATH}")
print(f"Training rows: {len(train_df):,}; columns: {train_df.shape[1]:,}")
print(f"Validation source: {VALIDATION_DATA_PATH}")
print(f"Validation rows: {len(validation_df):,}; columns: {validation_df.shape[1]:,}")


In [ ]:
def validate_source_frame(frame: pd.DataFrame, year: int, partition: str):
    """Validate one source population and return its audit summary."""
    required_audit_columns = {"FlightDate", "Origin", TARGET}
    missing_audit_columns = sorted(required_audit_columns - set(frame.columns))
    assert not missing_audit_columns, f"{partition} missing audit columns: {missing_audit_columns}"

    frame["FlightDate"] = pd.to_datetime(frame["FlightDate"], errors="coerce")
    assert frame["FlightDate"].notna().all(), f"{partition} has invalid FlightDate values"
    assert frame["FlightDate"].dt.year.eq(year).all(), f"{partition} has rows outside {year}"
    assert frame["Origin"].eq(AIRPORT).all(), f"{partition} Origin must equal {AIRPORT}"
    assert frame[TARGET].notna().all(), f"{partition} {TARGET} contains missing values"
    assert set(frame[TARGET].unique()).issubset({0, 0.0, 1, 1.0}), (
        f"{partition} has unexpected {TARGET} values"
    )

    return {
        "partition": partition,
        "year": year,
        "start": frame["FlightDate"].min().date(),
        "end": frame["FlightDate"].max().date(),
        "days": frame["FlightDate"].dt.normalize().nunique(),
        "rows": len(frame),
        "delayed_flights": int(frame[TARGET].sum()),
        "delay_rate": frame[TARGET].mean(),
    }


source_summary = pd.DataFrame(
    [
        validate_source_frame(train_df, TRAIN_YEAR, "training"),
        validate_source_frame(validation_df, VALIDATION_YEAR, "external validation"),
    ]
).set_index("partition")
source_summary


## Compact tree-oriented engineered feature manifest

The manifest uses representations that a bounded-depth decision tree can split directly or would benefit from receiving as a predefined summary:

1. **Calendar, schedule, and identity:** raw month, weekday, scheduled HHMM values, elapsed time, and distance remain because trees depend on ordering rather than linear spacing. Coarse time-of-day, weekend, scheduled-speed, and airline-destination features add useful groupings and one controlled interaction.
2. **Planned airport demand:** all six previous/current/next scheduled departure and arrival counts remain. Three-hour directional totals, total-traffic slopes, and the maximum hourly traffic expose demand level, direction, trend, and peak without requiring the tree to spend several levels reconstructing them.
3. **Weather:** raw temperature, humidity, visibility, and wind speed are paired with dew-point spread, log precipitation, wind components, condition count, adverse-weather flag, and observation age.

Deliberate exclusions keep the set smaller than the broad 54-feature pool: `ROUTE` duplicates destination for JFK departures; `AIRLINE_FLIGHT_ID` is high cardinality; sine/cosine pairs are less natural for a single tree; log distance preserves the same ordering as raw distance; and three-hour total traffic duplicates the two directional totals. Actual operational and delay-outcome fields remain forbidden.


In [ ]:
categorical_features = [
    "Month",
    "DayOfWeek",
    "Reporting_Airline",
    "Dest",
    "TIME_OF_DAY",
    "AIRLINE_DEST",
]

numeric_features = [
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "IS_WEEKEND",
    "SCHEDULED_SPEED_PROXY",
    "ASPM_PREVIOUS_SCHEDULED_DEPARTURES",
    "ASPM_PREVIOUS_SCHEDULED_ARRIVALS",
    "ASPM_CURRENT_SCHEDULED_DEPARTURES",
    "ASPM_CURRENT_SCHEDULED_ARRIVALS",
    "ASPM_NEXT_SCHEDULED_DEPARTURES",
    "ASPM_NEXT_SCHEDULED_ARRIVALS",
    "ASPM_THREE_HOUR_SCHEDULED_DEPARTURES",
    "ASPM_THREE_HOUR_SCHEDULED_ARRIVALS",
    "ASPM_CURRENT_MINUS_PREVIOUS_TRAFFIC",
    "ASPM_NEXT_MINUS_CURRENT_TRAFFIC",
    "ASPM_MAX_HOURLY_TRAFFIC",
    "HourlyDryBulbTemperature",
    "TEMP_DEWPOINT_SPREAD",
    "LOG_PRECIPITATION",
    "HourlyRelativeHumidity",
    "HourlyVisibility",
    "HourlyWindSpeed",
    "WindX",
    "WindY",
    "WEATHER_CONDITION_COUNT",
    "ADVERSE_WEATHER",
    "NOAA_AGE_MINUTES",
]

feature_columns = categorical_features + numeric_features
required_columns = ["FlightDate", "Origin", TARGET, *feature_columns]

assert len(feature_columns) == 34
assert len(feature_columns) == len(set(feature_columns)), "Duplicate feature names selected"
for partition, frame in [("training", train_df), ("external validation", validation_df)]:
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    assert not missing_columns, f"{partition} missing required columns: {missing_columns}"

forbidden_predictors = {
    TARGET,
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepartureDelayGroups",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrivalDelayGroups",
    "ActualElapsedTime",
    "AirTime",
    "Tail_Number",
}
leaking_columns = sorted(forbidden_predictors.intersection(feature_columns))
assert not leaking_columns, f"Prediction-time-invalid features selected: {leaking_columns}"

intentionally_excluded = {
    "ROUTE",
    "AIRLINE_FLIGHT_ID",
    "YEAR_PERIOD",
    "SCHED_DEP_TIME_SIN",
    "SCHED_DEP_TIME_COS",
    "SCHED_ARR_TIME_SIN",
    "SCHED_ARR_TIME_COS",
    "DAY_OF_WEEK_SIN",
    "DAY_OF_WEEK_COS",
    "DAY_OF_YEAR_SIN",
    "DAY_OF_YEAR_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "LOG_DISTANCE",
    "ASPM_THREE_HOUR_TOTAL_SCHEDULED_TRAFFIC",
}
unexpected_exclusions = sorted(intentionally_excluded.intersection(feature_columns))
assert not unexpected_exclusions, f"Intentionally excluded features selected: {unexpected_exclusions}"

print(f"Selected {len(feature_columns)} compact tree-oriented predictors")
print(f"  categorical: {len(categorical_features)}")
print(f"  numeric: {len(numeric_features)}")


## Missingness and temporal partitions

All 2019 rows form the training dataset. Its calendar days are divided into six consecutive blocks; five expanding-window folds train on all earlier blocks and validate on the next block. After selection, grid search refits the winning pipeline on all 2019 rows. The separately loaded 2023 rows are evaluated only after that refit.

No row is dropped for a missing predictor. Categorical mode imputation and numeric median imputation are learned separately inside every training fold. Numeric missingness indicators are added only when a training fold contains a missing value.

In [ ]:
def prepare_model_frame(source_frame: pd.DataFrame) -> pd.DataFrame:
    """Select columns, sort chronologically, and normalize input dtypes."""
    model_frame = source_frame[required_columns].copy()
    model_frame = model_frame.sort_values(
        ["FlightDate", "CRSDepTime"],
        kind="stable",
    ).reset_index(drop=True)
    model_frame[TARGET] = model_frame[TARGET].astype(int)

    for column in categorical_features:
        model_frame[column] = model_frame[column].astype(object)
    for column in numeric_features:
        model_frame[column] = pd.to_numeric(model_frame[column], errors="coerce")

    return model_frame


train_model_df = prepare_model_frame(train_df)
validation_model_df = prepare_model_frame(validation_df)

missing_summary = pd.DataFrame(
    {
        "training_missing": train_model_df[feature_columns].isna().sum(),
        "validation_missing": validation_model_df[feature_columns].isna().sum(),
    }
)
display(missing_summary[missing_summary.sum(axis=1).gt(0)])

X_train = train_model_df[feature_columns].reset_index(drop=True)
y_train = train_model_df[TARGET].reset_index(drop=True)
training_dates = train_model_df["FlightDate"].reset_index(drop=True)

X_validation = validation_model_df[feature_columns].reset_index(drop=True)
y_validation = validation_model_df[TARGET].reset_index(drop=True)
validation_dates = validation_model_df["FlightDate"].reset_index(drop=True)

assert training_dates.max() < validation_dates.min()


In [ ]:
def make_expanding_day_splits(dates: pd.Series, targets: pd.Series, n_splits: int):
    """Create expanding train/validation indices without splitting calendar days."""
    normalized_dates = dates.dt.normalize().to_numpy()
    unique_days = np.array(sorted(pd.unique(normalized_dates)))

    if len(unique_days) < n_splits + 1:
        raise ValueError("Not enough unique days for the requested temporal folds")

    day_blocks = np.array_split(unique_days, n_splits + 1)
    splits = []
    rows = []

    for fold_number in range(1, n_splits + 1):
        train_days = np.concatenate(day_blocks[:fold_number])
        validation_days = day_blocks[fold_number]
        train_indices = np.flatnonzero(np.isin(normalized_dates, train_days))
        validation_indices = np.flatnonzero(np.isin(normalized_dates, validation_days))

        assert train_indices.size and validation_indices.size
        assert train_days.max() < validation_days.min()
        splits.append((train_indices, validation_indices))
        rows.append(
            {
                "fold": fold_number,
                "train_start": pd.Timestamp(train_days.min()).date(),
                "train_end": pd.Timestamp(train_days.max()).date(),
                "validation_start": pd.Timestamp(validation_days.min()).date(),
                "validation_end": pd.Timestamp(validation_days.max()).date(),
                "train_rows": len(train_indices),
                "validation_rows": len(validation_indices),
                "train_delay_rate": targets.iloc[train_indices].mean(),
                "validation_delay_rate": targets.iloc[validation_indices].mean(),
            }
        )

    return splits, pd.DataFrame(rows).set_index("fold")


temporal_splits, fold_summary = make_expanding_day_splits(
    training_dates,
    y_train,
    N_TIME_SPLITS,
)

partition_summary = pd.DataFrame(
    {
        "start": [training_dates.min().date(), validation_dates.min().date()],
        "end": [training_dates.max().date(), validation_dates.max().date()],
        "days": [
            training_dates.dt.normalize().nunique(),
            validation_dates.dt.normalize().nunique(),
        ],
        "rows": [len(y_train), len(y_validation)],
        "delayed_flights": [y_train.sum(), y_validation.sum()],
        "delay_rate": [y_train.mean(), y_validation.mean()],
    },
    index=["2019 training", "2023 external validation"],
)

display(partition_summary)
fold_summary


## Preprocessing and controlled decision-tree search

Categorical mode imputation, one-hot encoding, numeric median imputation, and missingness indicators are part of the pipeline. Every learned transformation is therefore fitted only on the training portion of each temporal fold.

The grid is intentionally identical to Experiment 01:

- Gini and entropy split criteria;
- maximum depths of 5, 8, 10, and 15;
- minimum split sizes of 250 and 500;
- minimum leaf sizes of 100, 200, and 500;
- unweighted and balanced-class trees.

Holding the search fixed makes the comparison a test of the compact engineered representation rather than a larger tuning budget.


In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_features),
        ("numeric", numeric_pipeline, numeric_features),
    ],
    remainder="drop",
)

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(random_state=RANDOM_STATE),
        ),
    ]
)

parameter_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [5, 8, 10, 15],
    "classifier__min_samples_split": [250, 500],
    "classifier__min_samples_leaf": [100, 200, 500],
    "classifier__class_weight": [None, "balanced"],
}

scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "mcc": make_scorer(matthews_corrcoef),
    "neg_brier": "neg_brier_score",
}

grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="average_precision",
    cv=temporal_splits,
    n_jobs=N_JOBS,
    pre_dispatch=N_JOBS,
    verbose=1,
    return_train_score=False,
    error_score="raise",
)

parameter_count = len(list(ParameterGrid(parameter_grid)))
print(f"Hyperparameter combinations: {parameter_count:,}")
print(f"Temporal cross-validation fits: {parameter_count * len(temporal_splits):,}")


In [ ]:
search_start = perf_counter()
grid_search.fit(X_train, y_train)
search_seconds = perf_counter() - search_start

best_decision_tree_pipeline = grid_search.best_estimator_
fitted_tree = best_decision_tree_pipeline.named_steps["classifier"]

print(f"Grid search and refit completed in {search_seconds:,.1f} seconds")
print(f"Best mean temporal-validation average precision: {grid_search.best_score_:.4f}")
print(f"Refitted tree depth: {fitted_tree.get_depth()}")
print(f"Refitted terminal leaves: {fitted_tree.get_n_leaves():,}")
print("Best parameters:")
display(pd.Series(grid_search.best_params_, name="value").to_frame())


In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

result_columns = [
    "rank_test_average_precision",
    "mean_test_average_precision",
    "std_test_average_precision",
    "mean_test_roc_auc",
    "mean_test_balanced_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_mcc",
    "mean_test_neg_brier",
    "param_classifier__criterion",
    "param_classifier__max_depth",
    "param_classifier__min_samples_split",
    "param_classifier__min_samples_leaf",
    "param_classifier__class_weight",
]

display(
    cv_results.sort_values("rank_test_average_precision")[result_columns].head(12)
)

complexity_comparison = (
    cv_results.groupby(
        ["param_classifier__max_depth", "param_classifier__class_weight"],
        dropna=False,
    )
    .agg(
        best_mean_average_precision=("mean_test_average_precision", "max"),
        best_mean_roc_auc=("mean_test_roc_auc", "max"),
        best_mean_neg_brier=("mean_test_neg_brier", "max"),
        candidates=("mean_test_average_precision", "size"),
    )
    .sort_values("best_mean_average_precision", ascending=False)
)
complexity_comparison.head(12)


In [ ]:
encoded_feature_names = (
    best_decision_tree_pipeline.named_steps["preprocessor"].get_feature_names_out()
)

tree_summary = pd.Series(
    {
        "source_predictors": len(feature_columns),
        "encoded_predictors": len(encoded_feature_names),
        "tree_depth": fitted_tree.get_depth(),
        "terminal_leaves": fitted_tree.get_n_leaves(),
    },
    name="value",
)
tree_summary.to_frame()


## Select a classification threshold using 2019 only

Average precision and ROC AUC evaluate probability ranking, but operational labels require a threshold. The best pipeline is cloned and refitted once per temporal fold. Each validation row receives a probability from a model trained only on earlier days.

Thresholds from 0.05 through 0.50 are compared on these out-of-fold temporal predictions. The threshold with the highest F1 is selected, with balanced accuracy and then precision used only as tie-breakers. No 2023 label is used for threshold selection.

In [ ]:
oof_probabilities = np.full(len(y_train), np.nan)

for train_indices, validation_indices in temporal_splits:
    fold_model = clone(best_decision_tree_pipeline)
    fold_model.fit(
        X_train.iloc[train_indices],
        y_train.iloc[train_indices],
    )
    oof_probabilities[validation_indices] = fold_model.predict_proba(
        X_train.iloc[validation_indices]
    )[:, 1]

oof_mask = np.isfinite(oof_probabilities)
assert oof_mask.sum() > 0
assert np.isfinite(oof_probabilities[oof_mask]).all()

candidate_thresholds = np.round(np.arange(0.05, 0.501, 0.01), 2)
threshold_rows = []
oof_y = y_train.to_numpy()[oof_mask]
oof_p = oof_probabilities[oof_mask]

for threshold in candidate_thresholds:
    predictions = (oof_p >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": threshold,
            "accuracy": accuracy_score(oof_y, predictions),
            "balanced_accuracy": balanced_accuracy_score(oof_y, predictions),
            "precision": precision_score(oof_y, predictions, zero_division=0),
            "recall": recall_score(oof_y, predictions, zero_division=0),
            "f1": f1_score(oof_y, predictions, zero_division=0),
            "mcc": matthews_corrcoef(oof_y, predictions),
        }
    )

threshold_results = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_results.sort_values(
    ["f1", "balanced_accuracy", "precision"],
    ascending=False,
).iloc[0]
selected_threshold = float(best_threshold_row["threshold"])

print(f"Threshold-selection rows: {oof_mask.sum():,}")
print(f"Training-only selected threshold: {selected_threshold:.2f}")
display(
    threshold_results.sort_values(
        ["f1", "balanced_accuracy", "precision"],
        ascending=False,
    ).head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
for metric in ["precision", "recall", "f1", "balanced_accuracy", "mcc"]:
    ax.plot(threshold_results["threshold"], threshold_results[metric], label=metric)

ax.axvline(0.50, color="gray", linestyle="--", label="default threshold (0.50)")
ax.axvline(
    selected_threshold,
    color="black",
    linestyle=":",
    label=f"selected threshold ({selected_threshold:.2f})",
)
ax.set(
    title="Expanding-fold out-of-fold threshold comparison",
    xlabel="Classification threshold",
    ylabel="Score",
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## Evaluate the 2023 external validation dataset

The grid-selected estimator was refitted automatically on all 2019 training rows. It is now evaluated on 2023 at both the default 0.50 threshold and the threshold selected from 2019 out-of-fold probabilities. The class-prior dummy provides the no-skill reference.

Threshold changes affect label metrics such as precision and recall, but they do not change average precision, ROC AUC, or Brier score.

In [ ]:
def evaluate_probabilities(y_true, probabilities, model_name, threshold):
    """Return threshold, ranking, correlation, and calibration metrics."""
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "mcc": matthews_corrcoef(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
    }


validation_probabilities = best_decision_tree_pipeline.predict_proba(X_validation)[:, 1]

dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(X_train, y_train)
dummy_probabilities = dummy_model.predict_proba(X_validation)[:, 1]

evaluation_results = pd.DataFrame(
    [
        evaluate_probabilities(
            y_validation,
            dummy_probabilities,
            "Dummy (2019 class prior)",
            0.50,
        ),
        evaluate_probabilities(
            y_validation,
            validation_probabilities,
            "Decision tree (default threshold)",
            0.50,
        ),
        evaluate_probabilities(
            y_validation,
            validation_probabilities,
            "Decision tree (training-selected threshold)",
            selected_threshold,
        ),
    ]
).set_index("model")

evaluation_results


In [ ]:
default_predictions = (validation_probabilities >= 0.50).astype(int)
selected_predictions = (validation_probabilities >= selected_threshold).astype(int)

false_positive_rate, true_positive_rate, _ = roc_curve(y_validation, validation_probabilities)
curve_precision, curve_recall, _ = precision_recall_curve(y_validation, validation_probabilities)
calibration_predicted, calibration_observed = calibration_curve(
    y_validation,
    validation_probabilities,
    n_bins=10,
    strategy="quantile",
)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

ConfusionMatrixDisplay.from_predictions(
    y_validation,
    default_predictions,
    display_labels=["On time", "Delayed"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0, 0],
)
axes[0, 0].set_title("2023 validation: threshold 0.50")

ConfusionMatrixDisplay.from_predictions(
    y_validation,
    selected_predictions,
    display_labels=["On time", "Delayed"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0, 1],
)
axes[0, 1].set_title(f"2023 validation: threshold {selected_threshold:.2f}")

axes[1, 0].plot(
    false_positive_rate,
    true_positive_rate,
    label=f"ROC AUC = {roc_auc_score(y_validation, validation_probabilities):.3f}",
)
axes[1, 0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="No skill")
axes[1, 0].set(
    xlabel="False-positive rate",
    ylabel="True-positive rate",
    title="2023 external-validation ROC curve",
)
axes[1, 0].legend()

axes[1, 1].plot(
    curve_recall,
    curve_precision,
    label=f"AP = {average_precision_score(y_validation, validation_probabilities):.3f}",
)
axes[1, 1].axhline(
    y_validation.mean(),
    linestyle="--",
    color="gray",
    label=f"Prevalence = {y_validation.mean():.3f}",
)
axes[1, 1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="2023 external-validation precision-recall curve",
)
axes[1, 1].legend()

fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.plot(
    calibration_predicted,
    calibration_observed,
    marker="o",
    label="Decision tree",
)
ax.set(
    xlabel="Mean predicted probability",
    ylabel="Observed delayed-flight rate",
    title="2023 external-validation calibration",
)
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Inspect impurity-based feature importance

The fitted tree reports the total impurity reduction attributed to each encoded predictor. The first table shows encoded columns; the second aggregates one-hot categories back to their source predictor. Impurity importance is descriptive rather than causal and can favor predictors with more possible split points.


In [ ]:
def source_feature_from_encoded_name(encoded_name: str) -> str:
    """Map a ColumnTransformer output name back to its source column."""
    transformer_name, transformed_name = encoded_name.split("__", 1)
    if transformer_name == "numeric":
        return transformed_name.removeprefix("missingindicator_")

    for source_name in sorted(categorical_features, key=len, reverse=True):
        if transformed_name == source_name or transformed_name.startswith(f"{source_name}_"):
            return source_name
    return transformed_name


importance_table = (
    pd.DataFrame(
        {
            "encoded_feature": encoded_feature_names,
            "importance": fitted_tree.feature_importances_,
        }
    )
    .assign(
        source_feature=lambda frame: frame["encoded_feature"].map(
            source_feature_from_encoded_name
        )
    )
    .sort_values("importance", ascending=False)
)

source_importance = (
    importance_table.groupby("source_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

display(importance_table.head(20))
source_importance.head(20)


## Visualize the first three tree levels

Only the top of the fitted tree is displayed. The full selected tree can contain many leaves and is not useful as a single static figure.


In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=encoded_feature_names,
    class_names=["On time", "Delayed"],
    max_depth=3,
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=8,
    ax=ax,
)
ax.set_title("Experiment 02 decision tree: first three levels")
plt.tight_layout()
plt.show()


## Experiment 02 interpretation checklist

Use the displayed results to compare the compact engineered tree with the raw Experiment 01 tree:

1. Did the engineered manifest improve both 2019 temporal-validation and 2023 external-validation average precision?
2. Did ROC AUC and Brier score improve as well as average precision?
3. Did the selected depth, leaf size, or class weighting change?
4. Did engineered congestion summaries receive material importance beyond their six raw ASPM inputs?
5. Did the training-selected threshold improve the 2023 precision/recall trade-off?
6. Does any gain justify carrying this manifest into Random Forest and CatBoost experiments?

Treat 2023 as an external development/validation year: its results must not be used to refit or tune this Experiment 02 record. Compare both decision-tree experiments using their identical folds, search grid, target population, validation rows, metrics, and threshold policy. The 2024 dataset remains locked for final evaluation.
